# Label Value Count Summary

This notebook summarizes counts of `1`, `0`, and missing/masked labels for each column in the `no_prop_overlap` CSV splits.

Missing/masked count includes:
- explicit mask value (`-1` by default)
- NA-like entries (`NA`, `na`, `NaN`, empty string)
- values that fail numeric coercion

It reports counts for each split (`train`, `val`, `test`) and for all splits combined.

In [1]:
from pathlib import Path
from typing import Iterable

import pandas as pd

# Configure mask handling to match training script conventions.
MASK_VALUE = -1.0
NA_STRINGS = ("NA", "na", "NaN", "")
SPLITS = ("train", "val", "test")
PREFIX = "no_prop_overlap_"

PROPAGANDA_COLUMNS = (
    "Loaded_Language", "Slogans", "Black-and-White_Fallacy", "Flag-Waving",
    "Name_Calling", "Whataboutism", "Causal_Oversimplification",
    "Exaggeration,Minimisation", "Doubt", "Appeal_to_Authority", "Repetition",
    "Appeal_to_fear-prejudice", "Red_Herring", "Thought-terminating_Cliches",
    "Reductio_ad_hitlerum", "Bandwagon",
    "Obfuscation,Intentional_Vagueness,Confusion", "Straw_Men",
)
WA_BI_COLUMN = "WA_bi"
DISCOURSE_BINARY_COLUMNS = (
    "disagree_agree_bi", "attacking_respectful_bi", "emotion_fact_bi", "nasty_nice_bi",
    "personal_audience_bi", "defeater_undercutter_bi", "negotiate_attack_bi",
    "questioning_asserting_bi",
)


def resolve_data_dir() -> Path:
    """Find the data directory from common working directories."""
    candidates = [
        Path.cwd() / "data",
        Path.cwd().parent / "data",
        Path.cwd().parent.parent / "data",
    ]
    for candidate in candidates:
        if (candidate / f"{PREFIX}train.csv").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find data directory with no_prop_overlap CSVs. "
        "Run notebook from repo root or cluster_runs/"
    )


def label_columns(df: pd.DataFrame) -> list[str]:
    """Treat all columns except text/dataset as label columns."""
    return [c for c in df.columns if c not in {"text", "dataset"}]


def grouped_columns(df: pd.DataFrame) -> dict[str, list[str]]:
    """Return the paper-replication label groups in fixed order."""
    available = set(df.columns)
    groups = {
        "propaganda": [c for c in PROPAGANDA_COLUMNS if c in available],
        "wa_bi": [WA_BI_COLUMN] if WA_BI_COLUMN in available else [],
        "discourse_binaries": [c for c in DISCOURSE_BINARY_COLUMNS if c in available],
    }
    return groups


def summarize_frame(df: pd.DataFrame, columns: Iterable[str], split_name: str) -> pd.DataFrame:
    rows = []
    for col in columns:
        raw = df[col]
        numeric = pd.to_numeric(raw.replace(list(NA_STRINGS), pd.NA), errors="coerce")

        is_mask = numeric.isna() | (numeric == MASK_VALUE)
        is_one = numeric == 1
        is_zero = numeric == 0
        is_other = (~is_mask) & (~is_one) & (~is_zero)

        rows.append(
            {
                "split": split_name,
                "column": col,
                "count_1": int(is_one.sum()),
                "count_0": int(is_zero.sum()),
                "count_mask": int(is_mask.sum()),
                "count_other": int(is_other.sum()),
                "total_rows": int(len(df)),
            }
        )

    out = pd.DataFrame(rows)
    out["pct_mask"] = (out["count_mask"] / out["total_rows"] * 100).round(2)
    return out.sort_values(["count_mask", "column"], ascending=[False, True]).reset_index(drop=True)


data_dir = resolve_data_dir()
paths = {split: data_dir / f"{PREFIX}{split}.csv" for split in SPLITS}

for split, p in paths.items():
    if not p.exists():
        raise FileNotFoundError(f"Missing split file: {p}")

print(f"Using data directory: {data_dir}")
print("Split files:")
for split, p in paths.items():
    print(f"  {split}: {p}")

Using data directory: /Users/jgint/Documents/code/multitask-imehlhaff/data
Split files:
  train: /Users/jgint/Documents/code/multitask-imehlhaff/data/no_prop_overlap_train.csv
  val: /Users/jgint/Documents/code/multitask-imehlhaff/data/no_prop_overlap_val.csv
  test: /Users/jgint/Documents/code/multitask-imehlhaff/data/no_prop_overlap_test.csv


In [2]:
# Build per-split summaries grouped by paper replication families
summaries_by_split: dict[str, pd.DataFrame] = {}
group_summaries_by_split: dict[str, dict[str, pd.DataFrame]] = {}

for split in SPLITS:
    df_split = pd.read_csv(paths[split])

    # Full summary across all label columns (for reference)
    cols = label_columns(df_split)
    summaries_by_split[split] = summarize_frame(df_split, cols, split)

    # Grouped summaries requested: propaganda, WA_bi, discourse binaries
    groups = grouped_columns(df_split)
    group_summaries_by_split[split] = {
        group_name: summarize_frame(df_split, group_cols, split)
        for group_name, group_cols in groups.items()
    }

# Combined summary over all splits
combined_df = pd.concat([pd.read_csv(paths[s]) for s in SPLITS], ignore_index=True)
combined_cols = label_columns(combined_df)
combined_summary = summarize_frame(combined_df, combined_cols, "all")

combined_group_summaries = {
    group_name: summarize_frame(combined_df, group_cols, "all")
    for group_name, group_cols in grouped_columns(combined_df).items()
}

print(f"Total label columns summarized: {len(combined_cols)}")
print("Grouped columns:")
for group_name, group_cols in grouped_columns(combined_df).items():
    print(f"  {group_name}: {len(group_cols)} columns")

Total label columns summarized: 37
Grouped columns:
  propaganda: 18 columns
  wa_bi: 1 columns
  discourse_binaries: 8 columns


In [3]:
# Show grouped split-by-split tables
for split in SPLITS:
    print(f"\n=== {split.upper()} ===")
    for group_name in ("propaganda", "wa_bi", "discourse_binaries"):
        print(f"\n--- {group_name} ---")
        display(group_summaries_by_split[split][group_name])

print("\n=== ALL SPLITS COMBINED ===")
for group_name in ("propaganda", "wa_bi", "discourse_binaries"):
    print(f"\n--- {group_name} ---")
    display(combined_group_summaries[group_name])


=== TRAIN ===

--- propaganda ---


,split,column,count_1,count_0,count_mask,count_other,total_rows,pct_mask
0,train,Appeal_to_Authority,433,61476,157105,0,219014,71.73
1,train,Appeal_to_fear-prejudice,876,61033,157105,0,219014,71.73
2,train,Bandwagon,35,61874,157105,0,219014,71.73
3,train,Black-and-White_Fallacy,438,61471,157105,0,219014,71.73
4,train,Causal_Oversimplification,781,61128,157105,0,219014,71.73
5,train,Doubt,1958,59951,157105,0,219014,71.73
6,train,"Exaggeration,Minimisation",1753,60156,157105,0,219014,71.73
7,train,Flag-Waving,954,60955,157105,0,219014,71.73
8,train,Loaded_Language,8155,53754,157105,0,219014,71.73
9,train,Name_Calling,4218,57691,157105,0,219014,71.73



--- wa_bi ---


,split,column,count_1,count_0,count_mask,count_other,total_rows,pct_mask
0,train,WA_bi,90143,5893,122978,0,219014,56.15



--- discourse_binaries ---


...

,split,column,count_1,count_0,count_mask,count_other,total_rows,pct_mask
0,all,Appeal_to_Authority,500,66541,166554,0,233595,71.3
1,all,Appeal_to_fear-prejudice,1040,66001,166554,0,233595,71.3
2,all,Bandwagon,42,66999,166554,0,233595,71.3
3,all,Black-and-White_Fallacy,472,66569,166554,0,233595,71.3
4,all,Causal_Oversimplification,837,66204,166554,0,233595,71.3
5,all,Doubt,2082,64959,166554,0,233595,71.3
6,all,"Exaggeration,Minimisation",1910,65131,166554,0,233595,71.3
7,all,Flag-Waving,1058,65983,166554,0,233595,71.3
8,all,Loaded_Language,8814,58227,166554,0,233595,71.3
9,all,Name_Calling,4547,62494,166554,0,233595,71.3



--- wa_bi ---


,split,column,count_1,count_0,count_mask,count_other,total_rows,pct_mask
0,all,WA_bi,94656,6191,132748,0,233595,56.83



--- discourse_binaries ---


,split,column,count_1,count_0,count_mask,count_other,total_rows,pct_mask
0,all,defeater_undercutter_bi,16254,9999,207342,0,233595,88.76
1,all,personal_audience_bi,19845,6757,206993,0,233595,88.61
2,all,negotiate_attack_bi,16061,12697,204837,0,233595,87.69
3,all,questioning_asserting_bi,10966,21303,201326,0,233595,86.19
4,all,nasty_nice_bi,18883,52074,162638,0,233595,69.62
5,all,attacking_respectful_bi,24098,47045,162452,0,233595,69.54
6,all,disagree_agree_bi,56669,15072,161854,0,233595,69.29
7,all,emotion_fact_bi,48821,33582,151192,0,233595,64.72


In [ ]:
# Optional: save CSV outputs for downstream use
# output_dir = Path.cwd() / "outputs"
# output_dir.mkdir(parents=True, exist_ok=True)

# for split in SPLITS:
#     # Full-table export
#     summaries_by_split[split].to_csv(output_dir / f"label_value_counts_{split}.csv", index=False)

#     # Group-specific exports
#     for group_name, table in group_summaries_by_split[split].items():
#         table.to_csv(output_dir / f"label_value_counts_{split}_{group_name}.csv", index=False)

# combined_summary.to_csv(output_dir / "label_value_counts_all.csv", index=False)
# for group_name, table in combined_group_summaries.items():
#     table.to_csv(output_dir / f"label_value_counts_all_{group_name}.csv", index=False)

# print(f"Saved CSV summaries in: {output_dir}")

Saved CSV summaries in: /Users/jgint/Documents/code/multitask-imehlhaff/cluster_runs/outputs
